In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


In [21]:
CSV_PATH = 'raw_car_dataset.csv'
df = pd.read_csv(CSV_PATH)


In [22]:
print("\n === initail head ===")
print(df.head())


 === initail head ===
    Price  Odometer_km  Doors  Accidents Location  Year
0  $1,500     137830.0    4.0          1     City  1998
1  4171.0          NaN    4.0          0    Rural  2016
2  $5,331     107302.0    4.0          0   Suburb  2014
3  1500.0     141838.0    4.0          1   Suburb  1999
4  1500.0          NaN    3.0          0     City  2022


In [23]:
print("\n=== initail info ===")
print(df.info())
print(df.shape)


=== initail info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Price        145 non-null    object 
 1   Odometer_km  138 non-null    float64
 2   Doors        138 non-null    float64
 3   Accidents    145 non-null    int64  
 4   Location     140 non-null    object 
 5   Year         145 non-null    int64  
dtypes: float64(2), int64(2), object(2)
memory usage: 6.9+ KB
None
(145, 6)


In [24]:
print("\n === missing values ===")
print(df.isnull().sum())


 === missing values ===
Price          0
Odometer_km    7
Doors          7
Accidents      0
Location       5
Year           0
dtype: int64


In [25]:
# clean target
df["Price"] = df["Price"].replace(r"[\$,]", "" , regex=True).astype(float)


In [26]:
#fix catagorical
df["Location"] = df["Location"].replace({"subrb": "suburb", "??": pd.NA})

In [27]:
# impute missing value
df["Odometer_km"] = df["Odometer_km"].fillna(df["Odometer_km"].median())
df["Doors"] = df["Doors"].fillna(df["Doors"].mode()[0])
df["Accidents"] = df["Accidents"].fillna(df["Accidents"].mode()[0])
df["Location"] = df["Location"].fillna(df["Location"].mode()[0])


In [28]:
#duplicate remove
before = df.shape
df = df.drop_duplicates()
after = df.shape
print(f"Dropped duplicate: {before},  {after}")

Dropped duplicate: (145, 6),  (140, 6)


In [29]:
#outlier
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - k *iqr
    upper = q3 + k -iqr
    return lower , upper
    

In [30]:
low_price, high_price = iqr_fun(df["Price"])
lower_odometer_km , high_odometer_km = iqr_fun(df["Odometer_km"])

In [31]:
low_price, high_price = iqr_fun(df["Price"])
lower_odometer_km , high_odometer_km = iqr_fun(df["Odometer_km"])

In [32]:
#one hod encoding
df = pd.get_dummies(df, columns=["Location"], drop_first=False, dtype="int")

In [33]:
#feature engineering no leakage
CURRENT_YEAR = 2026
df["CarAge"] = CURRENT_YEAR - df["Year"]
df["km_per_year"] = df["Odometer_km"] / np.where(df["CarAge"] == 0 , 1 , df["CarAge"])
df["is_urban"] = ((df["Location_City"] == 1) |(df["Location_Suburb"] == 1)).astype(int)
df["LogPrice"] = np.log1p(df["Price"])
print("\n successfully created")
print(df[["Year" , "CarAge", "Odometer_km" , "km_per_year",  "is_urban", "Price", "LogPrice"]].head())


 successfully created
   Year  CarAge  Odometer_km   km_per_year  is_urban   Price  LogPrice
0  1998      28     137830.0   4922.500000         1  1500.0  7.313887
1  2016      10     128548.0  12854.800000         0  4171.0  8.336151
2  2014      12     107302.0   8941.833333         1  5331.0  8.581482
3  1999      27     141838.0   5253.259259         1  1500.0  7.313887
4  2022       4     128548.0  32137.000000         1  1500.0  7.313887


In [34]:
# feature scaling only x
dont_scale = {"Price", "LogPrice"}
cols = df.select_dtypes(include=["int64", "float64"]).columns.to_list()
exclude = [c for c in df.columns if c.startswith("Location_")]+ ["is_urban"]
feature_scale = [c for c in cols if c not in dont_scale and c not in exclude]
scale = StandardScaler()
df[feature_scale] = scale.fit_transform(df[feature_scale])

In [41]:
print("\n === final head ===")
print(df.head())


 === final head ===
    Price  Odometer_km     Doors  Accidents      Year  Location_City  \
0  1500.0     0.088107  0.254091   0.316968 -1.686714              1   
1  4171.0    -0.068130  0.254091  -0.820867  0.794617              0   
2  5331.0    -0.425746  0.254091  -0.820867  0.518913              0   
3  1500.0     0.155570  0.254091   0.316968 -1.548862              0   
4  1500.0    -0.068130 -0.931668  -0.820867  1.621727              1   

   Location_Rural  Location_Subrb  Location_Suburb    CarAge  km_per_year  \
0               0               0                0  1.686714    -0.609073   
1               1               0                0 -0.794617     0.054986   
2               0               0                1 -0.518913    -0.272591   
3               0               0                1  1.548862    -0.581383   
4               0               0                0 -1.621727     1.669210   

   is_urban  LogPrice  
0         1  7.313887  
1         0  8.336151  
2         1

In [42]:
print("\n==== final info====")
print(df.info())


==== final info====
<class 'pandas.core.frame.DataFrame'>
Index: 140 entries, 0 to 139
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Price            140 non-null    float64
 1   Odometer_km      140 non-null    float64
 2   Doors            140 non-null    float64
 3   Accidents        140 non-null    float64
 4   Year             140 non-null    float64
 5   Location_City    140 non-null    int64  
 6   Location_Rural   140 non-null    int64  
 7   Location_Subrb   140 non-null    int64  
 8   Location_Suburb  140 non-null    int64  
 9   CarAge           140 non-null    float64
 10  km_per_year      140 non-null    float64
 11  is_urban         140 non-null    int64  
 12  LogPrice         140 non-null    float64
dtypes: float64(8), int64(5)
memory usage: 15.3 KB
None


In [38]:
print("\n ====final missing value ====")
print(df.isnull().sum())


 ====final missing value ====
Price              0
Odometer_km        0
Doors              0
Accidents          0
Year               0
Location_City      0
Location_Rural     0
Location_Subrb     0
Location_Suburb    0
CarAge             0
km_per_year        0
is_urban           0
LogPrice           0
dtype: int64


In [39]:
print(df.shape)

(140, 13)


In [40]:
Clean_PATH = "Clean_cars_dataset.CSV"
df.to_csv(Clean_PATH, index=False)